# B04 — Premise Parser Evaluation

Calls `/premises` for sampled problems and inspects:
- FOL structure correctness (quantifier, operator, arguments)
- Predicate schema — canonical names and detected aliases
- Predicate renames applied during canonicalization
- Verification status (AST-to-schema consistency)

Gold FOL from `premises-FOL` is shown for reference but uses short identifiers; the parser output uses natural-language predicate names — compare structure, not spelling.

## 1. Configuration

In [ ]:
import json
import random
from pathlib import Path
from pprint import pprint

import httpx
from IPython.display import Markdown, display

API_BASE = "https://api.iamphuckhang.dev"
TIMEOUT_SECONDS = 120.0
SAMPLE_SIZE = 10
RANDOM_SEED = 42

DATASET_PATH = Path("../datasets/exact/Logic_Based_Educational_Queries.json")

async def call_premises(premises: list[str]) -> dict:
    payload = {"premises": premises}
    async with httpx.AsyncClient(timeout=TIMEOUT_SECONDS) as client:
        response = await client.post(f"{API_BASE}/premises", json=payload)
        response.raise_for_status()
        return response.json()

## 2. Load and Sample Dataset

In [ ]:
dataset = json.loads(DATASET_PATH.read_text())
print(f"Total problems: {len(dataset)}")

random.seed(RANDOM_SEED)
sample = random.sample(dataset, SAMPLE_SIZE)

print(f"Sampled {len(sample)} problems")
print(f"Premise counts: {[len(item['premises-NL']) for item in sample]}")

## 3. Parse All Sampled Problems

In [ ]:
results = []
for i, item in enumerate(sample):
    print(f"Parsing problem {i+1}/{len(sample)} ({len(item['premises-NL'])} premises)...", end=" ")
    try:
        result = await call_premises(item["premises-NL"])
        results.append({"item": item, "result": result, "error": None})
        print("ok")
    except Exception as exc:
        results.append({"item": item, "result": None, "error": str(exc)})
        print(f"ERROR: {exc}")

ok = sum(1 for r in results if r["error"] is None)
print(f"\n{ok}/{len(results)} succeeded")

## 4. Inspect Each Problem

In [ ]:
def show_problem(idx: int, entry: dict) -> None:
    item = entry["item"]
    result = entry["result"]
    error = entry["error"]

    display(Markdown(f"---\n### Problem {idx+1}"))

    if error:
        print(f"  ERROR: {error}")
        return

    parsed = result["premises"]
    gold_fol = item["premises-FOL"]
    nl = item["premises-NL"]

    display(Markdown("**NL → Parsed FOL (parser output) | Gold FOL**"))
    max_len = max(len(parsed), len(gold_fol))
    for j in range(max_len):
        nl_text  = nl[j] if j < len(nl) else "—"
        parsed_fol = parsed[j]["fol"] if j < len(parsed) else "—"
        gold      = gold_fol[j] if j < len(gold_fol) else "—"
        print(f"  [{j+1}] NL:     {nl_text}")
        print(f"       Parsed: {parsed_fol}")
        print(f"       Gold:   {gold}")
        print()

for i, entry in enumerate(results):
    show_problem(i, entry)

## 5. Structural Match Summary

A *structural match* checks that the top-level FOL shape (quantifier type, operator) matches the gold. This is approximate — the gold uses short identifiers while the parser uses full names.

In [ ]:
import re

def top_shape(fol: str) -> str:
    """Return coarse structural label from a FOL string."""
    fol = fol.strip()
    if fol.startswith("∀"):
        inner = fol.split(".", 1)[-1].strip()
        if "IMPLIES" in inner or "→" in inner or "⇒" in inner:
            return "FORALL-IMPLIES"
        if "AND" in inner or "∧" in inner:
            return "FORALL-AND"
        if "NOT" in inner or "¬" in inner:
            return "FORALL-NOT"
        return "FORALL-ATOMIC"
    if fol.startswith("∃"):
        return "EXISTS-ATOMIC"
    if "IMPLIES" in fol:
        return "IMPLIES"
    if "AND" in fol:
        return "AND"
    if "NOT" in fol or fol.startswith("NOT"):
        return "NOT-ATOMIC"
    return "ATOMIC"

def gold_shape(fol: str) -> str:
    fol = fol.strip()
    if fol.startswith("∀"):
        inner = fol.split(".", 1)[-1].strip()
        if "→" in inner:
            return "FORALL-IMPLIES"
        if "∧" in inner:
            return "FORALL-AND"
        if "¬" in inner:
            return "FORALL-NOT"
        return "FORALL-ATOMIC"
    if fol.startswith("∃"):
        return "EXISTS-ATOMIC"
    return "ATOMIC"

total, matched = 0, 0
mismatches = []

for entry in results:
    if entry["error"]:
        continue
    parsed_list = entry["result"]["premises"]
    gold_list = entry["item"]["premises-FOL"]
    for j, (p, g) in enumerate(zip(parsed_list, gold_list)):
        ps = top_shape(p["fol"])
        gs = gold_shape(g)
        total += 1
        if ps == gs:
            matched += 1
        else:
            mismatches.append({
                "nl": entry["item"]["premises-NL"][j],
                "parsed_fol": p["fol"],
                "gold_fol": g,
                "parsed_shape": ps,
                "gold_shape": gs,
            })

print(f"Structural match: {matched}/{total} = {matched/total:.1%}")
print(f"\nMismatches ({len(mismatches)}):")
for m in mismatches:
    print(f"  NL:     {m['nl']}")
    print(f"  Parsed: {m['parsed_fol']}  [{m['parsed_shape']}]")
    print(f"  Gold:   {m['gold_fol']}  [{m['gold_shape']}]")
    print()

## 6. Predicate Schema and Canonicalization

Shows per-problem predicate schemas and any renames detected.

In [ ]:
# NOTE: /premises response does not include schema/renames directly.
# Call /premises and also display predicate names extracted from the AST.

def collect_predicates(ast: dict) -> list[tuple[str, int]]:
    """Walk an AST dict and collect (predicate_name, arity) pairs."""
    results = []
    if ast["type"] == "atomic":
        results.append((ast["predicate"]["name"], len(ast["arguments"])))
    elif ast["type"] == "quantified":
        results.extend(collect_predicates(ast["body"]))
        if ast.get("restrictor"):
            results.extend(collect_predicates(ast["restrictor"]))
    elif ast["type"] == "logical":
        results.extend(collect_predicates(ast["left"]))
        if ast.get("right"):
            results.extend(collect_predicates(ast["right"]))
    return results

display(Markdown("### Predicate vocab per problem"))
for i, entry in enumerate(results):
    if entry["error"]:
        continue
    preds = {}
    for p in entry["result"]["premises"]:
        for name, arity in collect_predicates(p["ast"]):
            preds[f"{name}/{arity}"] = preds.get(f"{name}/{arity}", 0) + 1
    print(f"Problem {i+1}: {dict(sorted(preds.items()))}")

## 7. Export Raw Results

In [ ]:
output_path = Path("artifacts/reports/b04_premise_parser_eval.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

export = [
    {
        "gold_fol": entry["item"]["premises-FOL"],
        "nl": entry["item"]["premises-NL"],
        "parsed": entry["result"]["premises"] if entry["result"] else None,
        "error": entry["error"],
    }
    for entry in results
]
output_path.write_text(json.dumps(export, indent=2, ensure_ascii=False))
print(output_path.resolve())